# Centipede: test suite on a Kaggle GPU

This notebook runs the project's tests on a Kaggle T4, where MuJoCo Warp can use
its default Newton solver. It copies no project code: it clones the public
repository, installs the package as in the README, and runs `pytest`.

Before running, in the notebook's side panel:

- **Accelerator:** GPU T4. The P100 cannot compile the Newton solver.
- **Internet:** on (needed for `git clone` and `pip`).

Then choose **Run all** and keep the output of every cell.

In [ ]:
# The machine: GPU model, driver, and Python version.
!nvidia-smi
!python --version

In [ ]:
# Fetch the project. Change BRANCH to test another branch.
REPOSITORY = "https://github.com/DrippingSoup22/centipede-project.git"
BRANCH = "main"

!rm -rf /kaggle/working/centipede
!git clone --depth 1 --branch {BRANCH} {REPOSITORY} /kaggle/working/centipede
%cd /kaggle/working/centipede
!git log -1 --format="Testing commit %h: %s"

In [ ]:
# Install the package with the GPU backend and the test tools. Kaggle already
# provides PyTorch with CUDA, so it is not downloaded again.
!pip install --quiet -e ".[gpu,dev]"

In [ ]:
# The versions in use, and the GPU as PyTorch and Warp see it.
import mujoco
import mujoco_warp
import torch
import warp

print("MuJoCo", mujoco.__version__, "| MuJoCo Warp", mujoco_warp.__version__)
print("Warp", warp.__version__, "| PyTorch", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(),
    "| compute capability",
    torch.cuda.get_device_capability(),
)
assert torch.cuda.get_device_capability() >= (7, 0), "Choose the T4 accelerator"

In [ ]:
# The full test suite. On this GPU the GPU tests use the Newton solver.
!python -m pytest -q --durations=10